# Interactive Visual Analytics using Folium

In [1]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon

In [2]:
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
spacex_df = pd.read_csv(url)

In [3]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.

In [4]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

In [5]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

In [6]:
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
# Circle
for index,row in launch_sites_df.iterrows():
    lat = row['Lat']
    long = row['Long']
    launch = row['Launch Site']
    circle_from_data = folium.Circle([lat,long],radius=1000,color='#c900f7',fill=True).add_child(folium.Popup(launch))
    site_map.add_child(circle_from_data)
# Marker
    marker_from_data = folium.map.Marker(
    [lat,long],icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0),html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % launch
                           )
    )
    site_map.add_child(marker_from_data)


In [7]:
site_map

*   Are all launch sites in proximity to the Equator line? No
*   Are all launch sites in very close proximity to the coast? Yes

Most of the launch sites are located in Florida, US.

In [8]:
marker_cluster = MarkerCluster()

In [9]:
# length = len(spacex_df)
# markers = [''] * length
# spacex_df['marker_color'] = markers

marker = []
for index,row in spacex_df.iterrows():
    class_ = row['class']
    if class_ == 1:
        marker.append('green')
    else:
        marker.append('red')
spacex_df['marker_color'] = marker

In [10]:
# Add marker_cluster to current site_map
site_map.add_child(marker_cluster)

for index, row in spacex_df.iterrows():
    lat = row['Lat']
    long = row['Long']
    launch = row['Launch Site']
    color_marker = row['marker_color']
    marker_suc_data = folium.map.Marker(
        [lat,long],icon=folium.Icon(
            color='white',
            icon_color=color_marker,
            icon_size=(20,20),
            icon_anchor=(0,0))
    )
    marker_cluster.add_child(marker_suc_data)
site_map

In [11]:
# TASK 3: Calculate the distances between a launch site to its proximities

In [12]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

In [13]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

*TODO:* Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.

In [14]:
launch_site_lat = 28.56319
launch_site_lon = -80.57683
coastline_lat = 28.56367
coastline_lon = -80.56798

distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
distance_coastline

0.8662199482939856

In [15]:
distance_marker = folium.Marker(
    [coastline_lat,coastline_lon],
    icon = DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline)
    )
)
site_map.add_child(distance_marker)

lines = folium.PolyLine(locations=[[coastline_lat,coastline_lon],[launch_site_lat,launch_site_lon]],weight=1)
site_map.add_child(lines)

In [16]:
# Create a marker with distance to a closest city, railway, highway, etc.
# Draw a line between the marker to the launch site

# Launch site coordinates (CCAFS LC-40)
launch_site_lat = 28.56319
launch_site_lon = -80.57683

# Coordinates for nearest targets
points_of_interest = {
    'Highway': {'lat': 28.56372, 'lon': -80.57084, 'color': 'green'},
    'Railway': {'lat': 28.57217, 'lon': -80.58528, 'color': 'orange'},
    'City (Titusville)': {'lat': 28.61221, 'lon': -80.80756, 'color': 'purple'}

}

for name, poi in points_of_interest.items():
    # Calculate distance
    distance = calculate_distance(launch_site_lat, launch_site_lon, poi['lat'], poi['lon'])
    
    # Add distance text marker
    distance_marker = folium.Marker(
        [poi['lat'], poi['lon']],
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html=f'<div style="font-size: 11pt; color:{poi["color"]}; font-weight: bold;"><b>{distance:.2f} KM ({name})</b></div>'
        )
    )
    site_map.add_child(distance_marker)
    
    # Draw PolyLine between Launch Site and target
    lines = folium.PolyLine(
        locations=[[launch_site_lat, launch_site_lon], [poi['lat'], poi['lon']]],
        weight=2,
        color=poi['color']
    )
    site_map.add_child(lines)

# Render map
site_map

*   Are launch sites in close proximity to railways?
*   Are launch sites in close proximity to highways?
*   Are launch sites in close proximity to coastline?
*   Do launch sites keep certain distance away from cities?

**Railways:** Yes, launch sites are located in close proximity to railways to facilitate heavy transport logistics (such as moving rocket stages, boosters, engines, and heavy payloads directly to the assembly and launch facilities).

**Highways:** Yes, launch sites are near major highways and access roads to allow easy road transport of personnel, equipment, hardware, and supplies.

**Coastline:** Yes, launch sites are strategically situated right on or extremely close to coastlines. Launching over open ocean minimizes safety risks to populated inland regions in case of launch anomalies or trajectory failures, while also providing convenient maritime access for autonomous drone ship recovery operations.

**Cities:** Yes, launch sites intentionally maintain a significant safety buffer distance away from densely populated cities to protect urban populations from noise pollution, launch vibrations, toxic propellant exhaust, and potential catastrophic launch explosions or debris fall.

# Interactive Dashboards with Plotly

In [17]:
import pandas as pd
import dash
from dash import html
from dash import dcc
from dash.dependencies import Input,Output
import plotly.express as px

In [18]:
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv'

spacex_df = pd.read_csv(url)
spacex_df.head(2)

,Unnamed: 0,Flight Number,Launch Site,class,Payload Mass (kg),Booster Version,Booster Version Category
0,0,1,CCAFS LC-40,0,0.0,F9 v1.0 B0003,v1.0
1,1,2,CCAFS LC-40,0,0.0,F9 v1.0 B0004,v1.0


In [19]:
max_payload = spacex_df['Payload Mass (kg)'].max()
min_payload = spacex_df['Payload Mass (kg)'].min()

In [20]:
# Initialize Dash app
app = dash.Dash(__name__)

In [21]:
# APP LAYOUT
app.layout = html.Div(children= [
                      html.H1('SpaceX Launch Records Dashboard',
                              style = {
                                  'textAlign':'center',
                                  'color':'#503D36',
                                  'font-size':40
                              }
                             ),
                      # TASK 1: Add a Lauch Site Drop-down Input component
                      dcc.Dropdown(
                          id='site-dropdown',
                          options=
                              [{'label':'All Sites', 'value':'ALL'}] +
                              [{'label':site,'value':site}
                               for site in spacex_df['Launch Site'].unique()]
                          ,
                          value='ALL',
                          placeholder='Select a Launch Site here',
                          searchable=True
                      ),
                      html.Br(),

                      #TASK 2: Pie Chart - will be called in callback
                      html.Div(dcc.Graph(id='success-pie-chart')),
                      html.Br(),

                      html.P('Payload range (Kg):'),

                      #TASK 3: Add a RANGE SLIDER to Select Payload
                      dcc.RangeSlider(
                          id='payload-slider',
                          min=0,
                          max=10000,
                          step=1000,
                          marks={
                              0:'0',
                              2500:'2500',
                              5000:'5000',
                              7500:'7500',
                              10000:'10000'
                          },
                          value=[min_payload,max_payload]
                      ),
                      html.Br(),

                      #TASK 4: Scatter Chart - will be called in callback
                      html.Div(dcc.Graph(id='success-payload-scatter-chart')),
])                        

In [22]:
# CALLBACKS

#TASK 2: Add a callback function to render success pie-chart
#based on selected site dropdown

@app.callback(
    Output(
        component_id='success-pie-chart',
        component_property='figure'
    ),
    Input(
        component_id='site-dropdown',
        component_property='value'
    )
)
def get_pie_chart(entered_site):
    if entered_site == 'ALL':
        fig = px.pie(
            spacex_df,
            values='class',
            names='Launch Site',
            title='Total Success Launches by Launch Site'
        )
        return fig
    else:
        data = spacex_df[spacex_df['Launch Site']==entered_site]
        site_counts=data['class'].value_counts().reset_index()
        site_counts.columns = ['class','count']
        fig = px.pie(
            site_counts,
            values='count',
            names='class',
            title=f'Total Success Launches by {entered_site}'
        )
        return fig

#TASK 4: Add a callback function to render the success-payload-scatter-chart
@app.callback(
    Output(
        component_id='success-payload-scatter-chart',
        component_property='figure'
    ),
    [
        Input(
            component_id='site-dropdown',
            component_property='value'
        ),
        Input(
            component_id='payload-slider',
            component_property='value'
        )
    ]
)
def get_scatter_chart(entered_site,payload_range):
    filtered_df = spacex_df[spacex_df['Payload Mass (kg)'].between(payload_range[0],payload_range[1])]
    if entered_site == 'ALL':
        fig = px.scatter(
            filtered_df,
            x='Payload Mass (kg)',
            y='class',
            color='Booster Version Category'
        )
        return fig
    else:
        filtered_launch = filtered_df[filtered_df['Launch Site']==entered_site]
        fig = px.scatter(
            filtered_launch,
            x='Payload Mass (kg)',
            y='class',
            color='Booster Version Category'
        )
        return fig
    

In [23]:
# Execute the app
if __name__ == '__main__':
    app.run(debug=True,jupyter_mode='external')

Dash app running on http://127.0.0.1:8050/
